In [3]:
#Web Scraping Lab

import requests
from bs4 import BeautifulSoup
import pandas as pd


def scrape_books(min_rating, max_price):
    base_url = "http://books.toscrape.com/catalogue/page-{}.html"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    books_data = []

    for page in range(1, 51):  # Looping through the pages (adjust if site has fewer pages)
        response = requests.get(base_url.format(page), headers=headers)
        if response.status_code != 200:
            break

        soup = BeautifulSoup(response.content, "html.parser")
        books = soup.find_all("article", class_="product_pod")

        for book in books:
            # Extract rating
            rating = book.find("p", class_="star-rating")
            rating_text = rating["class"][1] if rating else "0"
            rating_mapping = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
            rating_value = rating_mapping.get(rating_text, 0)

            if rating_value < min_rating:
                continue

            # Extract book URL
            book_url = book.find("h3").find("a")["href"]
            book_url = f"http://books.toscrape.com/catalogue/{book_url}"

            # Visit book page for more details
            book_response = requests.get(book_url, headers=headers)
            book_soup = BeautifulSoup(book_response.content, "html.parser")

            # Extract data
            title = book_soup.find("h1").text.strip()
            price = float(book_soup.find("p", class_="price_color").text[1:])
            if price > max_price:
                continue

            availability = book_soup.find("p", class_="instock availability").text.strip()
            upc = book_soup.find("th", text="UPC").find_next("td").text.strip()
            genre = book_soup.find("ul", class_="breadcrumb").find_all("li")[2].text.strip()
            description_tag = book_soup.find("meta", attrs={"name": "description"})
            description = description_tag["content"].strip() if description_tag else "No description available."

            # Add to the data list
            books_data.append({
                "UPC": upc,
                "Title": title,
                "Price (£)": price,
                "Rating": rating_value,
                "Genre": genre,
                "Availability": availability,
                "Description": description
            })

    # Convert to DataFrame
    df = pd.DataFrame(books_data)
    return df

# Run the function with specified parameters
min_rating = 4
max_price = 20
books_df = scrape_books(min_rating, max_price)

# Save or display the data
books_df.to_csv("books_data.csv", index=False)
print(books_df.head())


C:\Users\30116822\AppData\Local\Temp\ipykernel_18844\2388030456.py:49: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  upc = book_soup.find("th", text="UPC").find_next("td").text.strip()


                UPC                                              Title  \
0  ce6396b0f23f6ecc                                        Set Me Free   
1  6258a1f6a6dcfe50  The Four Agreements: A Practical Guide to Pers...   
2  6be3beb0793a53e7                                     Sophie's World   
3  657fe5ead67a7767            Untitled Collection: Sabbath Poems 2014   
4  51653ef291ab7ddc                                    This One Summer   

   Price (£)  Rating           Genre             Availability  \
0      17.46       5     Young Adult  In stock (19 available)   
1      17.66       5    Spirituality  In stock (18 available)   
2      15.94       5      Philosophy  In stock (18 available)   
3      14.27       4          Poetry  In stock (16 available)   
4      19.49       4  Sequential Art  In stock (16 available)   

                                         Description  
0  Aaron Ledbetter’s future had been planned out ...  
1  In The Four Agreements, don Miguel Ruiz reveal...  